# LARA quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pfekin/LARA/blob/main/examples/quickstart.ipynb)

**Lightweight Additive Residual Adaptation**: residual-stream adapters for frozen
language models, routed per token.

This notebook trains two behaviors with two different objectives, then puts both
on one frozen model and routes between them. The base model is never modified.

**Runtime:** pick a GPU runtime (Runtime, Change runtime type, T4).

**Time:** about 20 minutes for the first cell, 5 for the second, 1 for the third,
plus a 3 GB model download on first use.

Run the cells in order. Cell 3 uses what cells 1 and 2 write to disk.


## Setup

In [ ]:
!pip install -q git+https://github.com/pfekin/LARA.git
!pip install -q transformers datasets accelerate trl

import gc
import torch
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          DataCollatorForLanguageModeling, Trainer, TrainingArguments)
from trl import DPOConfig, DPOTrainer

from lara import LARA, Bank

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU: pick a GPU runtime")

## 1. Train a behavior with cross-entropy

The only LARA-specific lines are the three marked below. Everything else is
ordinary Hugging Face training: LARA does not wrap, replace, or subclass the
trainer. It attaches modules to the model and freezes everything else, so
`model.parameters()` reaches the modules and any trainer picks them up.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          DataCollatorForLanguageModeling, Trainer, TrainingArguments)

from lara import LARA

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
OUT = "behaviors/code"
MAX_LEN = 512

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, device_map="auto")

# ── LARA (1/3): attach modules to the frozen base ────────────────────────────
# `layers=6` spreads six modules evenly over the depth. On a 28-layer model that
# is [4, 8, 12, 16, 20, 24]. Pass a list if you want exact control.
lara = LARA(model, layers=6, rank=128, alpha=128)
print(lara, f"\nbase frozen, {lara.num_trainable():,} trainable parameters")

raw = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train[:2000]")


def to_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tok.apply_chat_template(msgs, tokenize=False)}


def tokenize(ex):
    return tok(ex["text"], truncation=True, max_length=MAX_LEN)


ds = raw.map(to_text).map(tokenize, batched=True, remove_columns=raw.column_names)

# ── any trainer works: LARA only requires that the base stays frozen
trainer = Trainer(
    model=model,                      # the model itself; LARA rides along inside it
    args=TrainingArguments(
        output_dir="runs/code", max_steps=600, learning_rate=2e-4,
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        logging_steps=50, save_strategy="no", bf16=True, report_to=[],
    ),
    train_dataset=ds,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
)
trainer.train()

# ── LARA (2/3): save the behavior
# `route_samples` are short texts typical of this behavior. A Bank uses them to
# fit its router later, so the behavior can be shared and still routed without
# anyone needing this training set again.
#
# Use the outputs, not the instructions. The instructions in this corpus are
# English questions about code, so a router fitted on them would be separating
# two sets of English prose and could latch onto something incidental. The
# outputs are the code itself, which is what distinguishes this behavior.
samples = [ex["output"] for ex in raw.select(range(200))]
lara.save(OUT, route_samples=samples, method="ce")
print(f"wrote {OUT}")

# put the model back into an inference state before generating
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

# ── LARA (3/3): the scale is a runtime knob, no retraining
prompt = tok.apply_chat_template(
    [{"role": "user", "content": "Write a function that reverses a linked list."}],
    tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors="pt").to(model.device)

# gamma is continuous, but greedy decoding takes an argmax at every step, so
# nearby values often decode to identical text. The two ends show the shift.
# To see the middle, read the logits (below) or sample instead of decoding greedily.
for g in (0.0, 1.0):
    lara.gamma = g                    # 0.0 is the untouched base
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=80, do_sample=False)
    print(f"\n─── gamma={g} " + "─" * 40)
    print(tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))

# the knob is continuous underneath: next-token entropy moves with gamma even
# where the greedy path does not
print()
for g in (0.0, 0.25, 0.5, 0.75, 1.0):
    lara.gamma = g
    with torch.no_grad():
        lp = torch.log_softmax(model(**ids).logits[0, -1], dim=-1)
    print(f"gamma={g:<5} top={tok.decode(lp.argmax())!r:<12} "
          f"logprob={lp.max():.3f}  entropy={-(lp.exp() * lp).sum():.3f}")

# free the base before the next cell loads its own copy
lara.detach()
del model, trainer
gc.collect(); torch.cuda.empty_cache()


## 2. Align a behavior with DPO

A different objective, the same three LARA lines. The artifact this produces is
indistinguishable from the one above: a directory of projections that a bank can
route alongside any other behavior.

This is not a reproduction of the paper. The paper's DPO numbers come from the
harness in `research/`, which uses its own loss with an NLL anchor term that TRL
does not apply. Read the numbers below as a check that the modules trained, not
as a result.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer

from lara import LARA

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
OUT = "behaviors/polite"

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, device_map="auto")

# ── LARA (1/3): attach
# Preference optimization is less demanding about placement than fine-tuning:
# a single middle module reaches parity with six. `layers=1` resolves to [14]
# on a 28-layer model.
lara = LARA(model, layers=1, rank=128, alpha=128)
print(lara, f"\nbase frozen, {lara.num_trainable():,} trainable parameters")

ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs[:512]")


def to_pairs(ex):
    # Conversational format. TRL applies the chat template itself, so the join
    # between prompt and completion falls on a special token rather than gluing
    # two pieces of raw text together, where the tokenizer would merge across
    # the seam. It also matches the format the instruct model was trained in.
    return {"prompt": [{"role": "user", "content": ex["prompt"]}],
            "chosen": [{"role": "assistant", "content": ex["chosen"][-1]["content"]}],
            "rejected": [{"role": "assistant", "content": ex["rejected"][-1]["content"]}]}


ds = ds.map(to_pairs, remove_columns=ds.column_names)

# The reference model is the frozen base. Rather than load a second copy of it,
# precompute the reference log probabilities before training starts: at that
# point the modules are still zero-initialized, so the model *is* the base.
# (That holds when training a fresh behavior, as here. If you continue training
# an existing one, the modules are no longer zero and this shortcut is wrong.)
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        output_dir="runs/polite", max_steps=60, learning_rate=2e-4,
        per_device_train_batch_size=1, gradient_accumulation_steps=16,
        beta=5.0, max_length=512, precompute_ref_log_probs=True,
        # No warmup or decay schedule here. Both were tried: over 60 steps the
        # warmup ramp raises the loss for the first third of the run and the
        # held out shift came out lower than with a constant rate. The training
        # curve is noisy at this batch size, which is expected, and the number
        # worth reading is the shift on held out pairs below.
        logging_steps=5, save_strategy="no", bf16=True, report_to=[],
    ),
    train_dataset=ds,
    processing_class=tok,
)
trainer.train()

# ── LARA (2/3): save, same artifact shape as any other behavior
# The route samples are the preferred responses, not the prompts. A behavior is
# characterized by the text it produces, and the router reads the stream while
# that text is being generated.
lara.save(OUT, route_samples=[ex["chosen"][0]["content"] for ex in ds.select(range(200))],
          method="dpo")
print(f"wrote {OUT}")

# DPOConfig turns gradient checkpointing on and the KV cache off, and the trainer
# leaves the model in training mode. Generating in that state produces garbage,
# so put the model back into an inference state first.
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

# ── LARA (3/3): what DPO actually moved
# The margin between chosen and rejected is what DPO optimizes, and it is
# continuous. Reward accuracy thresholds it at zero, so a real shift in the
# margin can leave accuracy untouched if no pair crosses over.
import torch.nn.functional as F

eval_ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized",
                       split="test_prefs[:128]").map(to_pairs, remove_columns=None)


@torch.no_grad()
def seq_logprob(prompt, completion, normalize=False):
    p = tok(prompt, return_tensors="pt").input_ids.to(model.device)
    full = tok(prompt + completion, return_tensors="pt").input_ids.to(model.device)
    logits = model(full).logits[:, :-1]
    tgt = full[:, 1:]
    lp = torch.log_softmax(logits.float(), -1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    lp = lp[:, p.shape[1] - 1:]                     # completion tokens only
    return (lp.mean() if normalize else lp.sum()).item()


# Evaluate in the same format the adapter was trained in: the chat template up
# to the assistant turn is the prompt, the response text is the completion.
pairs = []
for ex in eval_ds:
    p = tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
    pairs.append((p, ex["chosen"][0]["content"], ex["rejected"][0]["content"]))

# Chosen responses in UltraFeedback are longer than rejected ones, and every
# token adds a negative log probability, so a summed margin mostly measures
# length: its accuracy comes out below chance. Normalizing by length is what
# makes the preference signal visible, which is why the paper's harness does it.
# The shift between gamma=0 and gamma=1 is the quantity to read here.
stats = {}
for g in (0.0, 1.0):
    lara.gamma = g
    summed = [seq_logprob(p, c) - seq_logprob(p, r) for p, c, r in pairs]
    normed = [seq_logprob(p, c, normalize=True) - seq_logprob(p, r, normalize=True)
              for p, c, r in pairs]
    stats[g] = (sum(summed) / len(summed), sum(normed) / len(normed),
                sum(m > 0 for m in normed) / len(normed))
    print(f"gamma={g}  margin/token {stats[g][1]:+.4f}  accuracy {stats[g][2]:.3f}  "
          f"(summed {stats[g][0]:+.2f}, length dominated)")

print(f"shift from the adapter: {stats[1.0][1] - stats[0.0][1]:+.4f} per token, "
      f"{stats[1.0][0] - stats[0.0][0]:+.3f} summed")

# a sample generation, for a qualitative look
prompt = tok.apply_chat_template(
    [{"role": "user", "content": "My code crashed again. What now?"}],
    tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors="pt").to(model.device)

for g in (0.0, 1.0):
    lara.gamma = g
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=80, do_sample=False)
    print(f"\n─── gamma={g} " + "─" * 40)
    print(tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))

# free the base before the next cell loads its own copy
lara.detach()
del model, trainer
gc.collect(); torch.cuda.empty_cache()


## 3. Both behaviors on one frozen model

No training here. This loads what the two cells above wrote, fits a small router
over them, and generates.

Note what does not happen: no behavior is merged into the weights, nothing is
loaded or unloaded between prompts, and adding a behavior later would not touch
the others.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from lara import Bank

BASE = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, device_map="auto")

# ── behaviors go on the shared frozen base
bank = Bank(model, tok)
bank.add("code", "behaviors/code")        # trained with cross-entropy (01)
bank.add("polite", "behaviors/polite")    # trained with DPO (02)
print(bank)

# The router reads the frozen stream, so it needs each behavior's route samples,
# which the behaviors carry with them. Seconds on any GPU.
bank.fit_router(steps=300, verbose=True)

mb = sum(p.numel() * p.element_size() for s in bank.sets for p in s.parameters()) / 1e6
print(f"\n{len(bank)} behaviors resident for {mb:.1f} MB over one base")

prompt = tok.apply_chat_template(
    [{"role": "user", "content": "My script throws a KeyError. Can you help?"}],
    tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors="pt").to(model.device)


def show(label):
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=100, do_sample=False)
    print(f"\n─── {label} " + "─" * 40)
    print(tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))


# ── routed: the router decides per token, and can apply both at once
bank.top_k = None                 # blend all behaviors by weight
show("routed (soft)")
print("mean routing weight:", {k: round(v, 3) for k, v in bank.route_weights(ids.input_ids).items()})

# ── top_k caps the cost: only the k highest-weighted behaviors are applied
bank.top_k = 1                    # hard selection, one behavior per token
show("routed (top_k=1)")

# ── pin overrides the router entirely, for when you know what you want
with bank.pin("code"):
    show("pinned to code")

with bank.pin({"code": 1.0, "polite": 0.4}):
    show("code, with the aligned style at 0.4")

# ── the base is still in there, untouched 
with bank.disabled():
    show("frozen base")

# free the base
bank.detach()
del model
gc.collect(); torch.cuda.empty_cache()


## What to look at

**The scale is a runtime value.** Cell 1 prints the top token and entropy across
gamma. The greedy path quantizes what you see, but the distribution moves
continuously underneath.

**The objective does not matter to the artifact.** Cell 2 trains with DPO and
writes the same kind of directory as cell 1.

**Routing has limits, and pinning is the answer.** In cell 3 the router reads the
prompt, and both behaviors here take English prompts, so it cannot separate them
from the prompt alone. Behaviors that differ by input domain (math against
medical, say) route cleanly. Behaviors that differ by output style are better
selected explicitly with `bank.pin(...)`, which the cell demonstrates.

## Next

- Repository: https://github.com/pfekin/LARA
- The experiments behind the paper: `research.md` in the repository